# Загрузка клинических рекомендаций в базу MedAP (через бесплатную видеокарту)

**Как пользоваться — по порядку:**
1. Вверху: **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.
2. На ноутбуке заархивируй папку `клинреки` в один файл `клинреки.zip` и загрузи его в **корень** своего Google Диска («Мой диск»).
3. В ячейке «НАСТРОЙКИ» ниже вставь свой адрес базы (`DATABASE_PUBLIC_URL` из Railway).
4. Вверху: **Среда выполнения → Выполнить все**. Дальше всё само.

⚠️ Пока идёт загрузка через Colab — **не запускай** параллельно загрузку на ноутбуке (чтобы не задваивать).

### Установка библиотек

In [ ]:
!pip -q install sentence-transformers pymupdf psycopg2-binary

### НАСТРОЙКИ — заполни это

In [ ]:
# 1) Адрес базы из Railway (Postgres -> Variables -> DATABASE_PUBLIC_URL). Целиком, с паролем:
DATABASE_URL = "postgresql://postgres:ПАРОЛЬ@ХОСТ.rlwy.net:ПОРТ/railway"

# 2) Имя zip-архива, который ты загрузил в корень Моего диска:
ZIP_NAME = "клинреки.zip"

### Проверка видеокарты

In [ ]:
import torch
assert torch.cuda.is_available(), \
    "GPU не включён! Среда выполнения -> Сменить среду выполнения -> T4 GPU, потом Выполнить все заново."
print("Видеокарта:", torch.cuda.get_device_name(0))

### Подключаем Google Диск и распаковываем архив

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, pathlib

zip_path = f"/content/drive/MyDrive/{ZIP_NAME}"
assert os.path.exists(zip_path), f"Не найден {zip_path}. Проверь имя архива и что он в корне Моего диска."
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/data')

CATS = ["взрослые", "дети", "взрослые_и_дети"]
root = None
for p in pathlib.Path('/content/data').rglob('*'):
    if p.is_dir() and any((p / c).is_dir() for c in CATS):
        root = p; break
assert root, "В архиве не нашлись подпапки взрослые/дети/взрослые_и_дети."
print("Папка с клинреками:", root)

### Загружаем поисковую модель (та же, что в боте)

In [ ]:
from sentence_transformers import SentenceTransformer
print("Скачиваю и загружаю модель (первый раз ~1-2 мин)...")
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda")
model.max_seq_length = 512
tok = model.tokenizer
print("Модель готова.")

### Извлечение текста, нарезка и эмбеддинги

In [ ]:
import fitz, re

CHUNK_TOKENS = 480   # < 512: гарантированно влезает в окно модели, ничего не обрезается
OVERLAP = 50
MIN_TOKENS = 20

def extract_pages(path):
    doc = fitz.open(path)
    pages = [pg.get_text() for pg in doc]
    doc.close()
    return pages

def chunk_pages(pages):
    out = []
    for pageno, text in enumerate(pages, 1):
        text = re.sub(r"\s+", " ", text or "").strip()
        if not text:
            continue
        ids = tok.encode(text, add_special_tokens=False)
        step = CHUNK_TOKENS - OVERLAP
        for s in range(0, len(ids), step):
            piece = ids[s:s + CHUNK_TOKENS]
            if len(piece) < MIN_TOKENS:
                break
            out.append((tok.decode(piece), pageno))
    return out

def embed(texts):
    v = model.encode(["passage: " + t for t in texts],
                     normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    return v.tolist()

### Загрузка в базу (возобновляемая: уже загруженные пропускаются)

In [ ]:
import psycopg2, traceback

url = DATABASE_URL.replace("postgresql+asyncpg://", "postgresql://")
conn = psycopg2.connect(url)
cur = conn.cursor()
cur.execute("SELECT title FROM books WHERE source_type = 'клинрек'")
existing = {r[0] for r in cur.fetchall()}
conn.commit()

files = []
for c in CATS:
    d = root / c
    if d.is_dir():
        files += [(p, c) for p in sorted(d.rglob("*.pdf"))]

total = len(files); ok = skip = err = 0
print(f"Всего файлов: {total}. Уже в базе: {len(existing)}.\n")

for i, (pdf, subj) in enumerate(files, 1):
    title = pdf.stem
    if title in existing:
        skip += 1
        print(f"skip ({i}/{total}) уже загружен: {pdf.name}")
        continue
    try:
        chunks = chunk_pages(extract_pages(str(pdf)))
        if not chunks:
            raise ValueError("не извлёкся текст")
        embs = embed([c for c, _ in chunks])
        cur.execute(
            "INSERT INTO books (title, author, subject, source_type, chunks_count) "
            "VALUES (%s, %s, %s, %s, %s) RETURNING id",
            (title, "", subj, "клинрек", len(chunks)))
        bid = cur.fetchone()[0]
        for idx, ((content, pageno), vec) in enumerate(zip(chunks, embs)):
            emb = "[" + ",".join(map(str, vec)) + "]"
            cur.execute(
                "INSERT INTO book_chunks (book_id, subject, author, title, source_type, "
                "chunk_index, content, embedding, page_from, page_to) "
                "VALUES (%s, %s, %s, %s, %s, %s, %s, %s::vector, %s, %s)",
                (bid, subj, "", title, "клинрек", idx, content, emb, pageno, pageno))
        conn.commit(); ok += 1
        print(f"OK   ({i}/{total}) {pdf.name} — {len(chunks)} чанков")
    except Exception as e:
        conn.rollback(); err += 1
        print(f"FAIL ({i}/{total}) {pdf.name}: {e}")

print(f"\nИТОГ: успешно {ok}, пропущено {skip}, ошибок {err} из {total}")
cur.close(); conn.close()